In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

def detect_shot_boundaries(video_path, hist_threshold=10.0, pixel_threshold=30.0, flow_threshold=1.0, max_frames=1000):
    cap = cv2.VideoCapture(video_path)
    prev_hist = None
    prev_frame = None
    shot_boundaries = []

    frame_count = 0
    hist_diffs = []
    pixel_diffs = []
    flow_magnitudes = []

    # Get the total number of frames in the video
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frames_to_process = min(total_frames, max_frames)

    # Initialize the progress bar
    with tqdm(total=frames_to_process, desc="Detecting Shot Boundaries") as pbar:
        while frame_count < frames_to_process:
            ret, frame = cap.read()
            if not ret:
                break

            # Convert frame to grayscale
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            hist = cv2.calcHist([gray], [0], None, [256], [0, 256])
            hist = cv2.normalize(hist, hist).flatten()

            if prev_hist is not None:
                # Histogram difference using Chi-Square method
                hist_diff = cv2.compareHist(prev_hist, hist, cv2.HISTCMP_CHISQR)
                hist_diffs.append(hist_diff)

                # Pixel-wise difference
                pixel_diff = np.mean(cv2.absdiff(prev_frame, gray))
                pixel_diffs.append(pixel_diff)

                # Optical Flow
                flow = cv2.calcOpticalFlowFarneback(prev_frame, gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)
                flow_magnitude = np.sqrt(flow[..., 0]**2 + flow[..., 1]**2).mean()
                flow_magnitudes.append(flow_magnitude)

                # Combining criteria for shot boundary
                if (hist_diff > hist_threshold) or (pixel_diff > pixel_threshold) or (flow_magnitude > flow_threshold):
                    shot_boundaries.append(frame_count)

            prev_hist = hist
            prev_frame = gray
            frame_count += 1

            # Update the progress bar
            pbar.update(1)

    cap.release()
    print(f"Detected {len(shot_boundaries)} shot boundaries.")

    # Plot differences for debugging
    plt.figure()
    plt.plot(hist_diffs, label='Histogram Difference')
    plt.plot(pixel_diffs, label='Pixel Difference')
    plt.plot(flow_magnitudes, label='Optical Flow Magnitude')
    plt.axhline(y=hist_threshold, color='r', linestyle='--', label='Hist Threshold')
    plt.axhline(y=pixel_threshold, color='g', linestyle='--', label='Pixel Threshold')
    plt.axhline(y=flow_threshold, color='b', linestyle='--', label='Flow Threshold')
    plt.legend()
    plt.title('Frame Differences and Thresholds')
    plt.xlabel('Frame')
    plt.ylabel('Difference / Magnitude')
    plt.show()

    return shot_boundaries

def extract_key_frames(shot_boundaries, video_path):
    cap = cv2.VideoCapture(video_path)
    key_frames = []
    frame_width = None
    frame_height = None

    with tqdm(total=len(shot_boundaries), desc="Extracting Key Frames") as pbar:
        for boundary in shot_boundaries:
            cap.set(cv2.CAP_PROP_POS_FRAMES, boundary)
            ret, frame = cap.read()
            if ret:
                if frame_width is None or frame_height is None:
                    frame_height, frame_width = frame.shape[:2]
                if frame.shape[1] == frame_width and frame.shape[0] == frame_height:
                    key_frames.append((boundary, frame))
                else:
                    print(f"Skipping frame at {boundary} due to size mismatch.")
            pbar.update(1)

    cap.release()
    print(f"Extracted {len(key_frames)} key frames.")
    return key_frames

def reconstruct_video(key_frames, output_path='output_compressed.mp4'):
    if not key_frames:
        print("No key frames available for reconstruction.")
        return

    frame_height, frame_width = key_frames[0][1].shape[:2]
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, 20.0, (frame_width, frame_height))
cosine
    # Initialize progress bar for reconstructing video
    with tqdm(total=len(key_frames), desc="Reconstructing Video") as pbar:
        for _, frame in key_frames:
            out.write(frame)
            pbar.update(1)

    out.release()
    print(f"Reconstructed video saved to {output_path}")

def compress_video(video_path, output_path='output_compressed.mp4', hist_threshold=10.0, pixel_threshold=30.0, flow_threshold=1.0, max_frames=1000):
    shot_boundaries = detect_shot_boundaries(video_path, hist_threshold, pixel_threshold, flow_threshold, max_frames)
    if shot_boundaries:
        key_frames = extract_key_frames(shot_boundaries, video_path)
        if key_frames:
            reconstruct_video(key_frames, output_path)
        else:
            print("No key frames detected. Cannot reconstruct video.")
    else:
        print("No shot boundaries detected. Cannot compress video.")

# Example usage
video_path = '/content/drive/MyDrive/rough/VideoCompession/AvengersEndgme.mkv'
compress_video(video_path, '/content/drive/MyDrive/rough/VideoCompession/compressed_output.mp4', hist_threshold=10.0, pixel_threshold=30.0, flow_threshold=1.0, max_frames=1000)


IndentationError: unexpected indent (<ipython-input-1-a11f87d5fbad>, line 110)